
# Scalar FD-BPM in Python

A compact educational implementation of the **scalar finite-difference Beam Propagation Method (FD-BPM)** using a Crank–Nicolson step.

The example is a **directional coupler**. We first calculate the fundamental mode of an isolated input waveguide, then use that mode as the input field for the BPM propagation through the two-waveguide structure.

The scalar BPM equation used here is
\[
\frac{\partial u}{\partial z}
=
-\frac{j}{2k_0n_{\rm ref}}
\frac{\partial^2 u}{\partial x^2}
-
j\frac{k_0}{2n_{\rm ref}}
\left[n^2(x)-n_{\rm ref}^2\right]u.
\]

The rapidly varying longitudinal phase has already been removed through
\[
E(x,z)=u(x,z)e^{-j\beta_{\rm ref}z},
\qquad
\beta_{\rm ref}=k_0n_{\rm ref}.
\]

This notebook intentionally uses a simple model:
- scalar BPM;
- 2D \(x\)-\(z\) geometry;
- weak/moderate index contrast;
- Dirichlet boundaries placed far from the guided field;
- no PML and no bidirectional propagation.

All distances are expressed in **micrometres**.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import diags, eye
from scipy.sparse.linalg import eigsh, factorized

plt.rcParams["figure.figsize"] = (8, 4.5)


## 1. Physical and numerical parameters

In [ ]:

# Optical parameters
lambda0 = 1.55          # vacuum wavelength [um]
k0 = 2*np.pi/lambda0

n_core = 1.50
n_clad = 1.45

# Directional-coupler geometry
width = 1.40            # waveguide width [um]
gap = 0.80              # edge-to-edge gap [um]

# Transverse computational window
W = 20.0                # total width [um]
Nx = 401
x = np.linspace(-W/2, W/2, Nx)
dx = x[1] - x[0]

# Longitudinal propagation
Lz = 400.0              # device length [um]
Nz = 1201
z = np.linspace(0.0, Lz, Nz)
dz = z[1] - z[0]

# Waveguide centres
x_left = -(gap/2 + width/2)
x_right = -x_left

print(f"dx = {dx:.4f} um")
print(f"dz = {dz:.4f} um")


## 2. Refractive-index profiles

In [ ]:

def single_waveguide_index(center):
    n = np.full_like(x, n_clad, dtype=float)
    n[np.abs(x-center) <= width/2] = n_core
    return n

def coupler_index():
    n = np.full_like(x, n_clad, dtype=float)
    left = np.abs(x-x_left) <= width/2
    right = np.abs(x-x_right) <= width/2
    n[left | right] = n_core
    return n

n_input = single_waveguide_index(x_left)
n_bpm = coupler_index()

plt.plot(x, n_input, label="isolated input waveguide")
plt.plot(x, n_bpm, "--", label="directional coupler")
plt.xlabel("x [um]")
plt.ylabel("refractive index")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



## 3. Fundamental mode of the isolated input waveguide

For a \(z\)-invariant scalar waveguide,
\[
\left[
\frac{d^2}{dx^2}+k_0^2n^2(x)
\right]U(x)
=
\beta^2U(x).
\]

We discretize the transverse operator with centred finite differences and solve the resulting sparse eigenvalue problem. The mode with the largest guided \(\beta\) is used as the BPM input.


In [ ]:

def fundamental_mode(n_profile):
    n_int = n_profile[1:-1]
    N = len(n_int)

    D2 = diags(
        [np.ones(N-1), -2*np.ones(N), np.ones(N-1)],
        offsets=[-1, 0, 1],
        format="csc"
    ) / dx**2

    H = D2 + diags((k0*n_int)**2, 0, format="csc")

    # A few eigenvalues near the top of the spectrum are sufficient here.
    eigenvalues, eigenvectors = eigsh(H, k=4, which="LA")

    i0 = np.argmax(eigenvalues)
    beta = np.sqrt(eigenvalues[i0])

    mode = np.zeros(Nx, dtype=complex)
    mode[1:-1] = eigenvectors[:, i0]

    # Normalize integral |u|^2 dx = 1
    mode /= np.sqrt(np.trapezoid(np.abs(mode)**2, x))

    # Arbitrary global sign: choose a positive real value near the guide centre.
    ic = np.argmin(np.abs(x-x_left))
    if np.real(mode[ic]) < 0:
        mode *= -1

    return beta, mode

beta0, u0 = fundamental_mode(n_input)
neff0 = beta0/k0

print(f"beta0 = {beta0:.6f} 1/um")
print(f"n_eff = {neff0:.6f}")

fig, ax1 = plt.subplots()
ax1.plot(x, np.abs(u0)**2, label=r"$|u_0|^2$")
ax1.set_xlabel("x [um]")
ax1.set_ylabel("normalized modal intensity")

ax2 = ax1.twinx()
ax2.plot(x, n_input, "--", alpha=0.6)
ax2.set_ylabel("refractive index")

plt.title("Fundamental mode of the isolated input waveguide")
plt.show()



## 4. Crank–Nicolson BPM operator

Write the propagation equation as
\[
\frac{\partial u}{\partial z}=\mathcal{L}u.
\]

The Crank–Nicolson step is
\[
\left(I-\frac{\Delta z}{2}\mathcal{L}\right)u^{m+1}
=
\left(I+\frac{\Delta z}{2}\mathcal{L}\right)u^m.
\]

For this directional coupler the cross-section does not change with \(z\), so the matrices are constant and can be factorized only once.

We choose
\[
n_{\rm ref}=n_{\rm eff}
\]
of the isolated input mode. This removes most of the rapid longitudinal phase of the injected guided field.


In [ ]:

nref = neff0

# Internal nodes only: u=0 is imposed at the two outer boundaries.
N = Nx - 2

D2 = diags(
    [np.ones(N-1), -2*np.ones(N), np.ones(N-1)],
    offsets=[-1, 0, 1],
    format="csc"
) / dx**2

# BPM operator L = A d2/dx2 + C(x)
A = -1j / (2*k0*nref)
C = -1j * k0/(2*nref) * (n_bpm[1:-1]**2 - nref**2)

L = A*D2 + diags(C, 0, format="csc")

I = eye(N, format="csc", dtype=complex)

M_left  = (I - dz*L/2).tocsc()
M_right = (I + dz*L/2).tocsc()

# The left-hand matrix is constant, so factorize it once.
solve_step = factorized(M_left)


## 5. BPM propagation

In [ ]:

u = u0[1:-1].copy()

# Store one field profile every few z steps to keep the output compact.
store_every = 3
z_store = []
field_store = []

for m in range(Nz):
    if m % store_every == 0:
        full = np.zeros(Nx, dtype=complex)
        full[1:-1] = u
        z_store.append(z[m])
        field_store.append(full)

    if m < Nz-1:
        rhs = M_right @ u
        u = solve_step(rhs)

z_store = np.asarray(z_store)
field_store = np.asarray(field_store).T

print("Propagation completed.")
print("Stored field array:", field_store.shape)


## 6. Field evolution

In [ ]:

intensity = np.abs(field_store)**2

plt.figure(figsize=(9, 4.8))
plt.imshow(
    intensity,
    extent=[z_store[0], z_store[-1], x[0], x[-1]],
    origin="lower",
    aspect="auto"
)
plt.xlabel("z [um]")
plt.ylabel("x [um]")
plt.title("FD-BPM propagation in a directional coupler")
plt.colorbar(label=r"$|u(x,z)|^2$")
plt.show()



## 7. Power exchange between the two guides

A simple qualitative estimate of the power in each branch is obtained by integrating \(|u|^2\) over the core region of each waveguide.


In [ ]:

mask_left = np.abs(x-x_left) <= width/2
mask_right = np.abs(x-x_right) <= width/2

P_left = np.trapezoid(
    np.abs(field_store[mask_left, :])**2,
    x[mask_left],
    axis=0
)

P_right = np.trapezoid(
    np.abs(field_store[mask_right, :])**2,
    x[mask_right],
    axis=0
)

plt.plot(z_store, P_left, label="left guide")
plt.plot(z_store, P_right, label="right guide")
plt.xlabel("z [um]")
plt.ylabel("power in core region [a.u.]")
plt.title("Directional-coupler power exchange")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

imax = np.argmax(P_right)
print(f"Maximum power in the right guide near z = {z_store[imax]:.1f} um")



## 8. Numerical checks

A smooth field map is **not** a convergence test. Repeat the simulation after changing:

- `Nx` to reduce \(\Delta x\);
- `Nz` to reduce \(\Delta z\);
- `W` to move the transverse boundaries farther away.

Monitor a physical quantity such as the coupling length or output power.

The simple Dirichlet condition used here is acceptable only because the computational window is chosen wide enough that the guided field is small at the boundaries.



## 9. Things to try

1. Change `gap` and observe the coupling length.
2. Change `width` and recompute the input mode.
3. Replace the directional coupler by a single straight guide and verify that the modal envelope remains almost unchanged.
4. Increase the index contrast and observe when the scalar/paraxial approximation becomes questionable.
5. Introduce a slowly varying geometry \(n(x,z)\). In that case the BPM matrices must be updated along \(z\).
6. Compare the result with a full-wave solver for a short structure containing an abrupt discontinuity.
